In [2]:
import pandas as pd

from fowt_ml import Config
from fowt_ml.datasets import get_data_mfiles, check_data, get_data
from fowt_ml.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from pathlib import Path
import numpy as np 
import torch

## 1 Model selection

### Create a config

In [3]:
# load the config
config_file = "./experiment_force_correction.yml"
config = Config.from_yaml(config_file)

In [4]:
# check model names in config 
config["ml_setup"]["model_names"]

{}

In [5]:
# set the model names; we selected three models
config["ml_setup"]["model_names"] = {
    "LinearRegression": {},
    
    "MultilayerPerceptron": {
        "hidden_layer_sizes": (8, 8, 8, 8, 8, 8),  
        "activation": "relu",                  
        "max_iter": 500,                     
        "alpha": 0.00001,                        
        "learning_rate": "adaptive",           
        "validation_fraction": 0.15,           
        "early_stopping": True,
        "n_iter_no_change": 15,              
        "random_state": 42
    },
    
    "XGBoostRegression": {
        "n_estimators": 500,           
        "max_depth": 10,                
        "learning_rate": 0.1,         
        "subsample": 0.8,
        "colsample_bytree": 0.8,       
        "colsample_bylevel": 0.7,      
        "min_child_weight": 5,         
        "gamma": 0.0,                  
        "tree_method": "hist",
        "random_state": 42
    }, 
}

In [6]:
# check train/test split kwargs
config["ml_setup"]["train_test_split_kwargs"]

{'train_size': 0.8, 'shuffle': False, 'random_state': 42}

In [7]:
# set shuffle to True because it is instantaneous prediction 
config["ml_setup"]["train_test_split_kwargs"] = {"train_size": 0.80, "shuffle": True, "random_state": 42}

In [8]:
data_folder = Path("./hybridlabs/data_x_wp2_202512/dataB/decay_cl_test")

# closed-lopp, surge, sway, heave, roll, pitch, yaw, no wind, no-rotation
data_ids = ["exp1644", "exp1646", "exp1648", "exp1651", "exp1653", "exp1656"]

### Compare models with different data

In [9]:
# get all data
df = get_data_mfiles(data_folder)
df.index = pd.to_timedelta(df.index*1e-3, unit='s')  # set time index 

# create holdout data (validation) with shuffle
data_traintest, data_validation = train_test_split(df, test_size=0.2, shuffle=True, random_state=42)
print(f"train size is {len(data_traintest)} and validation size is {len(data_validation)}.")
print("============================")


# create pipline and set it up
comparison_pipeline = Pipeline(config)
comparison_pipeline.setup(data=data_traintest)

# compare models
models, scores = comparison_pipeline.compare_models(sort="r2")
print(f"Scores of the models are:")
print(scores)
print("============================")

# validation on holdout data
best_model = scores.index[0]
model = models[best_model]
y_predict = model.predict(data_validation[comparison_pipeline.predictors_labels].values)
y_target = data_validation[comparison_pipeline.target_labels].values
r2 = r2_score(y_target, y_predict, multioutput='raw_values')
print(f"r2 of best model on validation set are: {r2}")


train size is 141844 and validation size is 35462.
Scores of the models are:
                      neg_mean_absolute_error        r2
XGBoostRegression                   -0.105725  0.660569
MultilayerPerceptron                -0.105815  0.572488
LinearRegression                    -0.108272  0.565529
r2 of best model on validation set are: [0.96838168 0.94677038 0.00777549 0.86711239 0.76231532 0.44804483]
